# House Price Prediction

**Goal:** Predict sale price of houses
**Algorithm:** Gradient Boosting Regressor

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor
%matplotlib inline

In [ ]:
np.random.seed(42)
n = 1000

sqft_living    = np.random.normal(2000, 500, n).astype(int)
bedrooms       = np.random.randint(1, 6, n)
bathrooms      = np.random.randint(1, 4, n)
floors         = np.random.randint(1, 3, n)
waterfront     = np.random.choice([0, 1], n, p=[0.9, 0.1])
year_built     = np.random.randint(1950, 2020, n)
condition      = np.random.randint(1, 6, n)
sqft_basement  = np.random.randint(0, 1000, n)

price = (sqft_living * 200 +
         bedrooms * 5000 +
         bathrooms * 10000 +
         waterfront * 100000 +
         condition * 15000 +
         (2020 - year_built) * (-500) +
         np.random.normal(0, 30000, n))
price = np.abs(price)

df = pd.DataFrame({
    'sqft_living': sqft_living,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'floors': floors,
    'waterfront': waterfront,
    'year_built': year_built,
    'condition': condition,
    'sqft_basement': sqft_basement,
    'price': price
})
print ('Shape: %s' % (df.shape,))
print ('First 5 rows:\n%s' % df.head())

<hr>## 1. Exploratory Data Analysis

In [ ]:
print ('Price statistics:\n%s' % df['price'].describe())
print ('\nMissing values: %s' % df.isnull().sum().sum())

<hr>## 2. Feature & Target Split

In [ ]:
X = df.drop('price', axis=1)
y = df['price']

# Log-transform (prices are skewed)
y_log = np.log1p(y)

print ('Before log: skew=%.2f' % y.skew())
print ('After log:  skew=%.2f' % y_log.skew())

<hr>## 3. Train Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42)
print ('Train: %s, Test: %s' % (X_train.shape[0], X_test.shape[0]))

In [ ]:
model = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    min_samples_leaf=5, random_state=42)
model.fit(X_train, y_train)
print ('Model: %s' % model)

<hr>## 4. Evaluate Performance

In [ ]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_actual = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
r2 = r2_score(y_actual, y_pred)

print ('Results (actual dollar values):')
print ('RMSE: $%.2f' % rmse)
print ('R²:   %.4f' % r2)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_actual, y_pred, alpha=0.5)
plt.plot([y_actual.min(), y_actual.max()],
         [y_actual.min(), y_actual.max()], 'r--', lw=2)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted House Prices')
plt.tight_layout()
plt.show()

<hr>## 5. Feature Importance

In [ ]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print ('Feature Importance:\n%s' % importances.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(importances['feature'], importances['importance'])
plt.xlabel('Importance')
plt.title('House Price - Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

<hr>## 6. Sample Predictions

In [ ]:
print ('Sample Predictions:')
for i in range(5):
    actual = y_actual.iloc[i]
    predicted = y_pred[i]
    diff = abs(actual - predicted)
    print ('Actual: $%10.2f | Predicted: $%10.2f | Diff: $%8.2f' % (actual, predicted, diff))